In [ ]:
# KrushiMitra — Plant Disease Detection Module (training & inference notebook)
# Attach dataset under DATA_ROOT (see config). Uncomment if needed:
# pip install tensorflow pillow numpy pandas scikit-learn matplotlib

from __future__ import annotations

import json
import logging
import os
import random
import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, BinaryIO, Iterable, Sequence, Union

import numpy as np

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("krushimitra.disease_detection")

PROJECT_ROOT = Path(os.environ.get("KRUSHIMITRA_ROOT", ".")).resolve()
DATA_ROOT = Path(os.environ.get("DISEASE_DATA_ROOT", PROJECT_ROOT / "data" / "plant_diseases"))
ARTIFACTS_DIR = Path(os.environ.get("DISEASE_ARTIFACTS_DIR", PROJECT_ROOT / "artifacts" / "disease_detection"))

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "valid"
TEST_DIR = DATA_ROOT / "test"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 1e-4
VAL_SPLIT = 0.2
SEED = 42

CONFIDENCE_THRESHOLD = 0.55
MIN_IMAGE_SIDE_PX = 64
MAX_IMAGE_SIDE_PX = 4096
BLUR_VARIANCE_THRESHOLD = 50.0

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
MODEL_FILENAME = "disease_classifier.keras"
LABEL_MAP_FILENAME = "label_map.json"
INFERENCE_CONFIG_FILENAME = "inference_config.json"

RUN_TRAINING = True
RUN_EVALUATION = True

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Data root: %s", DATA_ROOT)
logger.info("Artifacts: %s", ARTIFACTS_DIR)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from PIL import Image, ImageOps, UnidentifiedImageError

try:
    from sklearn.metrics import classification_report, confusion_matrix
except ImportError:
    classification_report = None
    confusion_matrix = None

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed()

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

logger.info("TensorFlow %s | GPUs: %d", tf.__version__, len(gpus))

In [ ]:
ImageInput = Union[str, Path, bytes, np.ndarray, Image.Image, BinaryIO]


@dataclass(frozen=True)
class ParsedLabel:
    class_id: int
    class_name: str
    crop: str
    condition: str
    is_healthy: bool


@dataclass
class PredictionItem:
    class_name: str
    crop: str
    condition: str
    is_healthy: bool
    probability: float


@dataclass
class DiseaseAnalysisResult:
    status: str
    message: str
    crop: str | None = None
    condition: str | None = None
    is_healthy: bool | None = None
    confidence: float | None = None
    top_predictions: list[PredictionItem] = field(default_factory=list)
    uncertainty_note: str | None = None
    image_quality: dict[str, Any] = field(default_factory=dict)
    metadata: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        d = asdict(self)
        d["top_predictions"] = [asdict(p) for p in self.top_predictions]
        return d


def parse_class_folder(name: str) -> tuple[str, str, bool]:
    """Parse folder names like Tomato___Late_blight or Apple___healthy."""
    parts = re.split(r"_{2,}|__", name.strip())
    if len(parts) >= 2:
        crop = parts[0].replace("_", " ").strip().title()
        condition_raw = " ".join(parts[1:]).replace("_", " ").strip()
    else:
        crop = name.replace("_", " ").strip().title()
        condition_raw = "unknown"
    healthy = condition_raw.lower() in {"healthy", "normal", "no disease"}
    condition = "Healthy" if healthy else condition_raw.title()
    return crop, condition, healthy


def build_label_map(class_names: Sequence[str]) -> list[ParsedLabel]:
    labels: list[ParsedLabel] = []
    for idx, name in enumerate(class_names):
        crop, condition, healthy = parse_class_folder(name)
        labels.append(
            ParsedLabel(
                class_id=idx,
                class_name=name,
                crop=crop,
                condition=condition,
                is_healthy=healthy,
            )
        )
    return labels


class ImagePreprocessor:
    """Normalize arbitrary farmer-uploaded images for model input."""

    def __init__(
        self,
        target_size: tuple[int, int] = IMAGE_SIZE,
        min_side: int = MIN_IMAGE_SIDE_PX,
        max_side: int = MAX_IMAGE_SIDE_PX,
    ) -> None:
        self.target_size = target_size
        self.min_side = min_side
        self.max_side = max_side

    def load_pil(self, source: ImageInput) -> Image.Image:
        if isinstance(source, Image.Image):
            img = source.convert("RGB")
        elif isinstance(source, (str, Path)):
            path = Path(source)
            if not path.exists():
                raise FileNotFoundError(f"Image not found: {path}")
            if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                raise ValueError(f"Unsupported extension: {path.suffix}")
            img = Image.open(path)
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
        elif isinstance(source, bytes):
            from io import BytesIO
            img = Image.open(BytesIO(source))
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
        elif isinstance(source, np.ndarray):
            arr = source
            if arr.ndim == 2:
                arr = np.stack([arr] * 3, axis=-1)
            elif arr.ndim == 3 and arr.shape[-1] == 4:
                arr = arr[..., :3]
            elif arr.ndim == 3 and arr.shape[-1] == 1:
                arr = np.repeat(arr, 3, axis=-1)
            if arr.dtype != np.uint8:
                arr = np.clip(arr, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr, mode="RGB")
        elif hasattr(source, "read"):
            img = Image.open(source)
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
        else:
            raise TypeError(f"Unsupported image source type: {type(source)!r}")
        return img

    def assess_quality(self, img: Image.Image) -> dict[str, Any]:
        w, h = img.size
        gray = np.array(img.convert("L"), dtype=np.float32)
        laplacian = (
            -4 * gray
            + np.roll(gray, 1, axis=0)
            + np.roll(gray, -1, axis=0)
            + np.roll(gray, 1, axis=1)
            + np.roll(gray, -1, axis=1)
        )
        blur_score = float(laplacian.var())
        return {
            "width": w,
            "height": h,
            "min_side": min(w, h),
            "blur_variance": blur_score,
            "likely_blurry": blur_score < BLUR_VARIANCE_THRESHOLD,
        }

    def resize_for_model(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        longest = max(w, h)
        if longest > self.max_side:
            scale = self.max_side / longest
            img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
        return img.resize(self.target_size, Image.Resampling.LANCZOS)

    def to_model_tensor(self, img: Image.Image) -> np.ndarray:
        arr = np.array(self.resize_for_model(img), dtype=np.float32)
        arr = np.expand_dims(arr, axis=0)
        return preprocess_input(arr)

    def validate(self, img: Image.Image) -> tuple[bool, str]:
        w, h = img.size
        if min(w, h) < self.min_side:
            return False, f"Image too small ({w}x{h}). Minimum side is {self.min_side}px."
        if w * h < 32 * 32:
            return False, "Image resolution is too low for reliable analysis."
        return True, "ok"

    def prepare(self, source: ImageInput) -> tuple[np.ndarray, dict[str, Any]]:
        try:
            pil_img = self.load_pil(source)
        except UnidentifiedImageError as exc:
            raise ValueError("File is not a readable image.") from exc
        ok, reason = self.validate(pil_img)
        if not ok:
            raise ValueError(reason)
        quality = self.assess_quality(pil_img)
        tensor = self.to_model_tensor(pil_img)
        return tensor, quality

In [ ]:
def resolve_dataset_directory() -> Path | None:
    candidates = [TRAIN_DIR, DATA_ROOT / "Train", DATA_ROOT]
    for path in candidates:
        if path.is_dir() and any(path.iterdir()):
            subdirs = [p for p in path.iterdir() if p.is_dir()]
            if subdirs and not any(p.suffix.lower() in SUPPORTED_EXTENSIONS for p in path.iterdir()):
                return path
    return None


def resolve_validation_directory(train_dir: Path) -> Path | None:
    if VAL_DIR.is_dir() and any(VAL_DIR.iterdir()):
        return VAL_DIR
    alt = train_dir.parent / "valid"
    return alt if alt.is_dir() and any(alt.iterdir()) else None


def create_datasets(train_dir: Path, val_dir: Path | None):
    common_kwargs = dict(
        labels="inferred",
        label_mode="categorical",
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
        interpolation="bilinear",
    )
    if val_dir:
        train_ds = keras.utils.image_dataset_from_directory(
            train_dir, shuffle=True, **common_kwargs
        )
        inferred_classes = train_ds.class_names
        val_ds = keras.utils.image_dataset_from_directory(
            val_dir, shuffle=False, class_names=inferred_classes, **common_kwargs
        )
    else:
        train_ds = keras.utils.image_dataset_from_directory(
            train_dir,
            validation_split=VAL_SPLIT,
            subset="training",
            shuffle=True,
            **common_kwargs,
        )
        inferred_classes = train_ds.class_names
        val_ds = keras.utils.image_dataset_from_directory(
            train_dir,
            validation_split=VAL_SPLIT,
            subset="validation",
            shuffle=False,
            **common_kwargs,
        )

    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=autotune)
    val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=autotune)
    train_ds = train_ds.prefetch(autotune)
    val_ds = val_ds.prefetch(autotune)
    return train_ds, val_ds, inferred_classes


def build_classifier(num_classes: int, fine_tune_at: int = 100) -> keras.Model:
    base = MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    model = keras.Model(inputs, outputs, name="krushimitra_disease_classifier")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )
    model._base_model = base
    model._fine_tune_at = fine_tune_at
    return model


def enable_fine_tuning(model: keras.Model, fine_tune_at: int | None = None) -> None:
    base = model._base_model
    at = fine_tune_at if fine_tune_at is not None else getattr(model, "_fine_tune_at", 100)
    base.trainable = True
    for layer in base.layers[:at]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

In [ ]:
class DiseaseDetector:
    """Production inference wrapper — accepts paths, bytes, PIL, or arrays."""

    def __init__(
        self,
        model: keras.Model,
        label_map: list[ParsedLabel],
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        top_k: int = 3,
    ) -> None:
        self.model = model
        self.label_map = label_map
        self.class_names = [lb.class_name for lb in label_map]
        self.confidence_threshold = confidence_threshold
        self.top_k = min(top_k, len(label_map))
        self.preprocessor = ImagePreprocessor()

    @classmethod
    def from_artifacts(cls, artifacts_dir: Path = ARTIFACTS_DIR) -> "DiseaseDetector":
        model_path = artifacts_dir / MODEL_FILENAME
        label_path = artifacts_dir / LABEL_MAP_FILENAME
        cfg_path = artifacts_dir / INFERENCE_CONFIG_FILENAME
        if not model_path.exists():
            raise FileNotFoundError(f"Trained model not found: {model_path}")
        if not label_path.exists():
            raise FileNotFoundError(f"Label map not found: {label_path}")
        model = keras.models.load_model(model_path)
        raw_labels = json.loads(label_path.read_text(encoding="utf-8"))
        label_map = [
            ParsedLabel(
                class_id=item["class_id"],
                class_name=item["class_name"],
                crop=item["crop"],
                condition=item["condition"],
                is_healthy=item["is_healthy"],
            )
            for item in raw_labels
        ]
        threshold = CONFIDENCE_THRESHOLD
        if cfg_path.exists():
            threshold = json.loads(cfg_path.read_text(encoding="utf-8")).get(
                "confidence_threshold", threshold
            )
        return cls(model=model, label_map=label_map, confidence_threshold=threshold)

    def _format_predictions(self, probs: np.ndarray) -> list[PredictionItem]:
        top_idx = np.argsort(probs)[::-1][: self.top_k]
        items: list[PredictionItem] = []
        for idx in top_idx:
            label = self.label_map[int(idx)]
            items.append(
                PredictionItem(
                    class_name=label.class_name,
                    crop=label.crop,
                    condition=label.condition,
                    is_healthy=label.is_healthy,
                    probability=float(probs[idx]),
                )
            )
        return items

    def analyze(self, source: ImageInput) -> DiseaseAnalysisResult:
        try:
            tensor, quality = self.preprocessor.prepare(source)
        except (FileNotFoundError, ValueError, TypeError) as exc:
            return DiseaseAnalysisResult(
                status="invalid_input",
                message=str(exc),
                metadata={"module": "disease_detection"},
            )

        probs = self.model.predict(tensor, verbose=0)[0]
        top_items = self._format_predictions(probs)
        best = top_items[0]
        confidence = best.probability

        uncertainty_note = None
        if quality.get("likely_blurry"):
            uncertainty_note = (
                "Image appears blurry. Retake a clear close-up of a single affected leaf in daylight."
            )
        if len(top_items) > 1 and (top_items[0].probability - top_items[1].probability) < 0.12:
            extra = "Top two predictions are close; treat result as tentative."
            uncertainty_note = f"{uncertainty_note} {extra}" if uncertainty_note else extra

        if confidence < self.confidence_threshold:
            return DiseaseAnalysisResult(
                status="low_confidence",
                message=(
                    "Could not identify disease reliably. "
                    "Use a clear photo of one leaf, fill the frame, and avoid shadows."
                ),
                crop=best.crop,
                condition=best.condition,
                is_healthy=best.is_healthy,
                confidence=round(confidence, 4),
                top_predictions=top_items,
                uncertainty_note=uncertainty_note,
                image_quality=quality,
                metadata={"module": "disease_detection"},
            )

        if best.is_healthy:
            msg = f"{best.crop} appears healthy with {confidence:.0%} confidence."
        else:
            msg = (
                f"Likely {best.condition} on {best.crop} ({confidence:.0%} confidence). "
                f"Confirm with local agronomist before treatment."
            )

        return DiseaseAnalysisResult(
            status="success",
            message=msg,
            crop=best.crop,
            condition=best.condition,
            is_healthy=best.is_healthy,
            confidence=round(confidence, 4),
            top_predictions=top_items,
            uncertainty_note=uncertainty_note,
            image_quality=quality,
            metadata={"module": "disease_detection"},
        )

    def analyze_batch(self, sources: Iterable[ImageInput]) -> list[dict[str, Any]]:
        return [self.analyze(src).to_dict() for src in sources]


def save_artifacts(
    model: keras.Model,
    label_map: list[ParsedLabel],
    metrics: dict[str, Any] | None = None,
    artifacts_dir: Path = ARTIFACTS_DIR,
) -> None:
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    model.save(artifacts_dir / MODEL_FILENAME)
    (artifacts_dir / LABEL_MAP_FILENAME).write_text(
        json.dumps([asdict(lb) for lb in label_map], indent=2),
        encoding="utf-8",
    )
    inference_cfg = {
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "image_size": IMAGE_SIZE,
        "supported_extensions": sorted(SUPPORTED_EXTENSIONS),
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": metrics or {},
    }
    (artifacts_dir / INFERENCE_CONFIG_FILENAME).write_text(
        json.dumps(inference_cfg, indent=2),
        encoding="utf-8",
    )
    logger.info("Artifacts saved to %s", artifacts_dir)

In [ ]:
train_dir = resolve_dataset_directory()
trained_model: keras.Model | None = None
parsed_labels: list[ParsedLabel] = []
history = None
val_metrics: dict[str, float] = {}

if train_dir is None:
    logger.warning(
        "No training data found. Place dataset under %s with class subfolders, then re-run.",
        TRAIN_DIR,
    )
elif RUN_TRAINING:
    val_dir = resolve_validation_directory(train_dir)
    train_ds, val_ds, class_names = create_datasets(train_dir, val_dir)
    parsed_labels = build_label_map(class_names)
    num_classes = len(class_names)
    logger.info("Classes: %d | Train dir: %s", num_classes, train_dir)

    trained_model = build_classifier(num_classes)
    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
        ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-6, monitor="val_loss"),
        ModelCheckpoint(
            filepath=str(ARTIFACTS_DIR / "checkpoint.keras"),
            save_best_only=True,
            monitor="val_accuracy",
        ),
    ]

    history = trained_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    enable_fine_tuning(trained_model)
    history_ft = trained_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max(8, EPOCHS // 3),
        callbacks=callbacks,
    )

    eval_result = trained_model.evaluate(val_ds, return_dict=True)
    val_metrics = {k: float(v) for k, v in eval_result.items()}
    logger.info("Validation metrics: %s", val_metrics)

    if RUN_EVALUATION and classification_report is not None:
        y_true, y_pred = [], []
        for batch_x, batch_y in val_ds:
            preds = trained_model.predict(batch_x, verbose=0)
            y_true.extend(np.argmax(batch_y.numpy(), axis=1))
            y_pred.extend(np.argmax(preds, axis=1))
        report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
        (ARTIFACTS_DIR / "classification_report.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        if confusion_matrix is not None:
            cm = confusion_matrix(y_true, y_pred)
            np.save(ARTIFACTS_DIR / "confusion_matrix.npy", cm)

    save_artifacts(trained_model, parsed_labels, metrics=val_metrics)
else:
    logger.info("RUN_TRAINING is False — skipping training.")

In [ ]:
# Inference: works after training or by loading saved artifacts
detector: DiseaseDetector | None = None

if trained_model is not None and parsed_labels:
    detector = DiseaseDetector(model=trained_model, label_map=parsed_labels)
elif (ARTIFACTS_DIR / MODEL_FILENAME).exists():
    detector = DiseaseDetector.from_artifacts(ARTIFACTS_DIR)
    logger.info("Loaded detector from artifacts.")
else:
    logger.warning("No model available for inference until dataset is trained.")


def analyze_image(path: str | Path) -> dict[str, Any]:
    if detector is None:
        raise RuntimeError("Detector not initialized. Train model or load artifacts first.")
    return detector.analyze(path).to_dict()


# Example: set SAMPLE_IMAGE to a leaf photo path after dataset/model is ready
SAMPLE_IMAGE = os.environ.get("DISEASE_SAMPLE_IMAGE", "")

if detector and SAMPLE_IMAGE:
    result = analyze_image(SAMPLE_IMAGE)
    print(json.dumps(result, indent=2))
elif detector:
    logger.info(
        "Set env DISEASE_SAMPLE_IMAGE=<path> or call analyze_image(path) to test inference."
    )